In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
# =========================================================================
# SCRIPT TRÍCH XUẤT TIỆN ÍCH BẤT ĐỘNG SẢN BẰNG QWEN 2.5 7B (CHẠY ĐỘC LẬP)
# =========================================================================

# 1. IMPORTS
import json
import re
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 2. KHỞI TẠO VÀ TẢI MÔ HÌNH QWEN 7B (NÉN 4-BIT)
model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("--- 1. Đang tải mô hình Qwen2.5-7B-Instruct ---")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("=> Đã tải xong Qwen2.5-7B-Instruct thành công và sẵn sàng.")

# 3. ĐỌC DỮ LIỆU
file_path = '/kaggle/input/datasets/locdovan211/project-bds-final/Processed_data.csv' 
print(f"\n--- 2. Đang đọc dữ liệu từ: {file_path} ---")
df = pd.read_csv(file_path)
print(f"=> Đã nạp file thành công! Kích thước dữ liệu: {df.shape}")

tien_ich_cols = [
    'hem_xe_hoi', 'gan_cho_sieu_thi', 'gan_truong_hoc', 
    'gan_benh_vien', 'gan_cong_vien_ho_nuoc'
]

# Khởi tạo các cột kết quả với giá trị mặc định là 0
for col in tien_ich_cols:
    if col not in df.columns:
        df[col] = 0

# 4. HÀM TRÍCH XUẤT
def extract_features_qwen(description, model, tokenizer):
    messages = [
        {"role": "system", "content": "Bạn là hệ thống trích xuất thông tin. Chỉ xuất kết quả định dạng JSON, tuyệt đối không giải thích văn bản."},
        {"role": "user", "content": f"""Đọc đoạn mô tả bất động sản sau và xác định xem các tiện ích xung quanh có được nhắc đến hay không.
Trả về giá trị 1 nếu CÓ nhắc đến (hoặc có từ viết tắt tương đương như: hxh, hẻm xe hơi, st, siêu thị, bv, bệnh viện, trg, trường học, cv, công viên), và 0 nếu KHÔNG nhắc đến.

BẮT BUỘC TRẢ VỀ ĐÚNG ĐỊNH DẠNG JSON VỚI CÁC KEY SAU:
{{
    "hem_xe_hoi": 0,
    "gan_cho_sieu_thi": 0,
    "gan_truong_hoc": 0,
    "gan_benh_vien": 0,
    "gan_cong_vien_ho_nuoc": 0
}}

Mô tả: "{description}" """}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs, 
            max_new_tokens=150, 
            do_sample=False
        )
        
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    
    try:
        if "{" in response:
            json_clean = re.search(r'\{.*\}', response, re.DOTALL).group()
            return json.loads(json_clean)
        return None
    except Exception:
        return {col: 0 for col in tien_ich_cols}

# 5. CHẠY VÒNG LẶP VÀ LƯU KẾT QUẢ
print("\n--- 3. Bắt đầu quét dữ liệu mô tả và trích xuất JSON ---")

for index, row in tqdm(df.iterrows(), total=len(df), desc="Tiến trình xử lý"):
    # LƯU Ý: Đảm bảo cột chứa văn bản trong file CSV của bạn tên là 'Mô tả'. 
    # Nếu tên khác (VD: 'description'), hãy sửa lại ở dòng dưới đây:
    description = row['Mô tả']  
    
    if pd.isna(description):
        continue
        
    extracted_data = extract_features_qwen(description, model, tokenizer)
    
    if extracted_data:
        for col in tien_ich_cols:
            df.at[index, col] = extracted_data.get(col, 0)

output_file = '/kaggle/working/Cleaned_data_qwen.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n=> 4. Hoàn thành xuất sắc! File kết quả đã được lưu tại: {output_file}")